# The Centerpoint Theorem and Tukey Depth

**A-Level Mathematics Extension: Geometry, Probability, and Robust Statistics**

---

## Welcome!

Imagine you are sending a team of robots to explore Mars. The robots take
position readings, but some sensors are faulty and give wildly wrong values.
How do you find a "central" meeting point that is robust to these outliers?

The usual **average** (mean) is famously sensitive to outliers. The **median**
in 1D is better — but generalising it to 2D or higher dimensions is surprisingly
subtle. This leads us to the beautiful **Centerpoint Theorem**.

## Section 1: Introduction — The Mars Robots Problem

### The Problem

You have $n$ robots at positions $p_1, p_2, \ldots, p_n$ in the plane.
You want to find a single point $c$ (the "rendezvous point") such that
**no halfplane** through $c$ contains too many robots on one side.

Why? Because if a halfplane has very few robots, that side of the plane
is "underrepresented" — meaning $c$ is not really in the "middle".

### The Centerpoint Theorem

> **Theorem (Rado 1947)**: For any finite set of $n$ points in $\mathbb{R}^d$,
> there exists a point $c$ (possibly not one of the data points) such that every
> closed halfspace through $c$ contains at least $\lceil n / (d+1) \rceil$ points.

In the plane ($d = 2$): every halfplane through $c$ contains at least $n/3$ points.

This point $c$ is called a **centerpoint**. It always exists! (But it might be
hard to find efficiently.)

### Three Notions of "Centre"

| Concept | Formula | Robust to outliers? |
|---------|---------|--------------------|
| Mean (average) | $\bar{p} = \frac{1}{n}\sum p_i$ | No — one outlier can move it arbitrarily |
| Coordinatewise median | Median of $x$-coords, median of $y$-coords | Partially — better but not $\geq 1/3$ guaranteed |
| Centerpoint | Exists by the theorem | Yes — always $\geq 1/(d+1)$ of points on each side |

## Section 2: Setup

We need NumPy for numerical computations and Matplotlib for visualisation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize

# Fix random seed for reproducibility
np.random.seed(2024)

print('Libraries loaded!')

## Section 3: The Average and Its Failure

Let's demonstrate how a single outlier can completely destroy the mean,
while the coordinatewise median remains much more stable.

In [ ]:
# Generate 50 random points in [0, 100]^2
np.random.seed(42)
normal_points = np.random.uniform(0, 100, size=(50, 2))

# Add one outlier far away
outlier = np.array([[1000, 1000]])
all_points = np.vstack([normal_points, outlier])

# Compute mean and coordinatewise median WITHOUT outlier
mean_clean = normal_points.mean(axis=0)
median_clean = np.median(normal_points, axis=0)

# Compute mean and coordinatewise median WITH outlier
mean_dirty = all_points.mean(axis=0)
median_dirty = np.median(all_points, axis=0)

print('Without outlier:')
print(f'  Mean   = ({mean_clean[0]:.1f}, {mean_clean[1]:.1f})')
print(f'  Median = ({median_clean[0]:.1f}, {median_clean[1]:.1f})')
print()
print('With outlier at (1000, 1000):')
print(f'  Mean   = ({mean_dirty[0]:.1f}, {mean_dirty[1]:.1f})  <-- moved a lot!')
print(f'  Median = ({median_dirty[0]:.1f}, {median_dirty[1]:.1f})  <-- barely moved!')

# ---- Plot ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (points, title, mean_pt, median_pt) in zip(axes, [
    (normal_points, 'Clean data (50 points)', mean_clean, median_clean),
    (all_points,   'Data with one outlier at (1000, 1000)', mean_dirty, median_dirty),
]):
    ax.scatter(points[:-1, 0], points[:-1, 1], color='steelblue', alpha=0.6,
               s=40, label='Normal points', zorder=2)
    if len(points) > 50:
        ax.scatter(points[-1, 0], points[-1, 1], color='red', s=150,
                   marker='*', label='Outlier', zorder=4)

    ax.scatter(*mean_pt, color='darkorange', s=200, marker='D',
               label=f'Mean ({mean_pt[0]:.0f}, {mean_pt[1]:.0f})', zorder=5)
    ax.scatter(*median_pt, color='green', s=200, marker='P',
               label=f'Median ({median_pt[0]:.0f}, {median_pt[1]:.0f})', zorder=5)

    ax.set_title(title, fontsize=11)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

fig.suptitle('Mean vs. Coordinatewise Median: Effect of One Outlier', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Key Observation

One single outlier at $(1000, 1000)$ dragged the mean from approximately
$(50, 50)$ to over $(60, 60)$ — a significant shift away from the bulk of the data.

The coordinatewise median, however, barely moved. This is the hallmark of
**robust statistics**: methods that give reliable answers even when some data
is corrupted or extreme.

But is the coordinatewise median actually a good "centre"? Let's see a counterexample.

## Section 4: The Five-Point Counterexample

Consider the five points: $(3,0), (4,0), (5,0), (0,3), (0,4)$.

The coordinatewise median is the median of the $x$-coordinates and
the median of the $y$-coordinates independently.

- $x$-values: $0, 0, 3, 4, 5$ → median $= 3$
- $y$-values: $0, 0, 3, 4, 5$ → median $= 3$ 

Wait — let's compute this carefully. The points are $(3,0), (4,0), (5,0), (0,3), (0,4)$:
- $x$-coords: $3, 4, 5, 0, 0$ → sorted: $0, 0, 3, 4, 5$ → median $= 3$
- $y$-coords: $0, 0, 0, 3, 4$ → sorted: $0, 0, 0, 3, 4$ → median $= 0$

So the coordinatewise median is $(3, 0)$ — which is actually one of the data points!
Is this a good centre? Let's check: how many points lie in each halfplane.

In [ ]:
# The five-point example
points_5 = np.array([
    [3, 0],
    [4, 0],
    [5, 0],
    [0, 3],
    [0, 4]
], dtype=float)

n5 = len(points_5)

# Coordinatewise median
coord_median_5 = np.median(points_5, axis=0)
print(f'Points: {points_5.tolist()}')
print(f'Coordinatewise median: {coord_median_5}')

# Consider the halfplane x >= 3  (everything to the right of x=3)
# The point (3,0) is ON the boundary, let's count strictly to the right
# For the halfplane defined by normal vector (1,0) through (3,0):
#   points with x > 3: (4,0) and (5,0) — that's 2 out of 5, which is 2/5 = 40%
# Now consider the halfplane to the LEFT: x < 3: (0,3) and (0,4) — also 2 out of 5

# The CRITICAL halfplane: above the line y = 0 (the x-axis)
# All points with y > 0: (0,3) and (0,4) — that's 2 out of 5 = 40%
# The OTHER halfplane y <= 0: (3,0), (4,0), (5,0), (0,0) — but (0,0) is not in our set
# Actually let's be precise: halfplane y > 0 contains {(0,3),(0,4)} = 2 points
# 2/5 = 0.4 > 1/3 = 0.333, so that's fine.
# But what about the halfplane containing ONLY the three bottom points?

# The halfplane y <= 0 but shifted to pass through coord_median = (3,0):
# Normal vector pointing up: (0,1), passing through (3,0)
# Points with y >= 0 (i.e., dot product with (0,1) >= 0 relative to (3,0)):
#   All points: y values are 0, 0, 0, 3, 4
#   y >= 0 for ALL five points. Hmm.

# The problem: halfplane x >= 3 through coord_median = (3,0)
# Points with x >= 3: (3,0), (4,0), (5,0) — that's 3 out of 5 (60%)
# Points with x < 3:  (0,3), (0,4) — that's 2 out of 5 (40%)

# The WORSE halfplane for coord_median:
# Consider the halfplane defined by the line through (3,0) with normal pointing
# toward (4,0) and (5,0) — i.e., the right halfplane.
# Points strictly to the LEFT (other side): only (0,3), (0,4) = 2 points = 2/5
# Hmm 2/5 > 1/3 still.

# Actually the real issue is a halfplane through a DIFFERENT boundary:
# Draw a line through (3,0) separating {(0,3),(0,4)} from {(4,0),(5,0)}.
# Such a line would leave only 2 points on one side — still 2/5.

# The true counterexample to coord_median: we need a halfplane leaving < n/3 points
# n/3 = 5/3 ≈ 1.67, so we need a halfplane with ≤ 1 point on one side.
# Does coord_median = (3,0) have a halfplane with ≤ 1 point on one side?

# Halfplane to the upper-left of (3,0), direction perpendicular to (3,3)->(0,0):
# Let's try the halfplane: x + y >= 3, passing through (3,0)
# Points satisfying x + y < 3: just (0,3)? No: 0+3=3. None! Let's try x + y <= 3:
# (3,0): 3+0=3 YES, (4,0): 4, NO, (5,0): 5, NO, (0,3): 0+3=3 YES, (0,4): 4 NO
# So halfplane {x+y<=3} contains (3,0) and (0,3) — 2 points. Other side has 3.

# For the coordinatewise median (3,0), the minimum fraction in any halfplane
# through it... let's compute this properly below.
# The point: coord_median may have < 1/3 fraction on one side.

# More dramatic: the halfplane containing ONLY (0,3) and (0,4) but not the median:
# e.g. line x = 1.5 (vertical), halfplane x <= 1.5
# Points with x <= 1.5: (0,3) and (0,4) — that's 2/5 = 40% > 1/3
# Hmm. Let me try to find a halfplane through (3,0) that has just 1 point on one side.

# Halfplane through (3,0) with direction tilted: normal = (1, 1)/sqrt(2)
# dot(p - (3,0), (1,1)) >= 0 means x + y >= 3
# Points: (3,0):3, (4,0):4, (5,0):5, (0,3):3, (0,4):4
# All satisfy x+y >= 3. The other side {x+y < 3} has 0 points.

# Rotate a bit: normal (1, -1), halfplane x - y >= 3
# Points: (3,0):3, (4,0):4, (5,0):5, (0,3):-3, (0,4):-4
# x-y >= 3: (3,0), (4,0), (5,0) — 3 points; other side: (0,3),(0,4) — 2 points

# The critical direction for coord_median: the open halfplane above-left
# Let's just show the Tukey depth of coord_median vs a proper centerpoint

print()
print('For the 5-point example, we will compute Tukey depth below.')
print('First, let us plot the points and the coordinatewise median.')

fig, ax = plt.subplots(figsize=(7, 6))

# Plot the 5 data points
ax.scatter(points_5[:, 0], points_5[:, 1], color='steelblue', s=150, zorder=5,
           label='Data points')
for i, (x, y) in enumerate(points_5):
    ax.annotate(f'$p_{i+1}$=({x:.0f},{y:.0f})', (x, y),
                textcoords='offset points', xytext=(8, 5), fontsize=10)

# Plot the coordinatewise median
ax.scatter(*coord_median_5, color='green', s=200, marker='P', zorder=6,
           label=f'Coord. median = ({coord_median_5[0]:.0f}, {coord_median_5[1]:.0f})')

# Draw a halfplane showing the potential issue:
# Halfplane: the half-space x <= 2 (everything to the left)
# This halfplane does NOT contain the coord_median (which is at x=3)
# but wait — we want a halfplane THROUGH the coord_median.
# Let's draw: line through (3,0) with slope 1 (direction (1,1))
# Halfplane above this line: y > -(x-3) i.e. x+y < 3
# This halfplane through (3,0) contains NO data points — showing (3,0) is on the boundary

# Instead show the halfplane y >= 0 (everything above x-axis, through the median (3,0))
# Points above: (0,3), (0,4) — 2 points; below: (3,0) itself, (4,0), (5,0)
# Draw the dividing line (x-axis through (3,0))
ax.axhline(y=0, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label='Halfplane boundary (y=0)')
ax.fill_between([-1, 7], 0, 6, alpha=0.1, color='orange',
                label='Upper halfplane (2 of 5 points = 40%)')

ax.set_xlim(-1, 7)
ax.set_ylim(-1, 6)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Five-point example\nCoordinatewise median and a halfplane through it', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print()
print('The upper halfplane (y >= 0) through the coord. median (3,0) contains:')
count_above = np.sum(points_5[:, 1] >= 0)
print(f'  {count_above} out of {n5} points = {count_above/n5:.2f}')
print(f'  This is {'≥' if count_above/n5 >= 1/3 else '<'} 1/3 = {1/3:.3f}')

## Section 5: Tukey Depth

The **Tukey halfspace depth** of a point $p$ with respect to a dataset $X$
is the *minimum fraction* of points that lie in any closed halfspace containing $p$.

$$\text{depth}(p, X) = \min_{\theta} \frac{\#\{x_i : x_i \cdot \theta \geq p \cdot \theta\}}{n}$$

where the minimum is over all unit direction vectors $\theta$.

- A point with **high Tukey depth** is genuinely central — no halfplane through it has too few points.
- The **centerpoint** is exactly the point with Tukey depth $\geq 1/(d+1) = 1/3$ in 2D.
- The **Tukey median** is the point of maximum depth.

We compute depth approximately by sweeping over 180 directions.

In [ ]:
def tukey_depth(point, data):
    """
    Approximate Tukey halfspace depth of a point with respect to a dataset.

    We sweep over 180 evenly spaced directions in [0, pi) and for each direction
    compute the fraction of data points that lie in the halfplane 'on the same side'
    as the point. The depth is the minimum fraction over all directions.

    Parameters
    ----------
    point : array-like, shape (2,)
        The query point.
    data : array-like, shape (n, 2)
        The dataset of n 2D points.

    Returns
    -------
    float
        Approximate Tukey depth of `point` w.r.t. `data`, as a fraction of n.
    """
    data = np.asarray(data, dtype=float)
    point = np.asarray(point, dtype=float)
    n = len(data)
    min_frac = 1.0

    # Sweep over 180 directions (half-circle is sufficient due to symmetry)
    for theta in np.linspace(0, np.pi, 180):
        # Unit direction vector
        d = np.array([np.cos(theta), np.sin(theta)])

        # Project all data points and the query point onto direction d
        projs = data @ d         # shape (n,)
        p_proj = np.dot(d, point)

        # Count how many points are on each side
        count_leq = np.sum(projs <= p_proj)   # on the 'left' of the query
        count_geq = np.sum(projs >= p_proj)   # on the 'right' of the query

        # The fraction in the smaller halfplane
        frac = min(count_leq, count_geq) / n
        min_frac = min(min_frac, frac)

    return min_frac


# Quick test on the 5-point example
depth_coord_median = tukey_depth(coord_median_5, points_5)
print(f'Tukey depth of coordinatewise median {coord_median_5} = {depth_coord_median:.3f}')
print(f'  Is depth >= 1/3? {depth_coord_median >= 1/3 - 1e-9}')
print(f'  1/3 = {1/3:.3f}')

# Test a few other points
test_pts = [(2, 1), (1, 2), (1.5, 1.5), (0, 0)]
print()
print('Depths of other candidate points:')
for pt in test_pts:
    d = tukey_depth(np.array(pt, dtype=float), points_5)
    print(f'  depth({pt}) = {d:.3f}  {"<-- candidate centerpoint!" if d >= 1/3 - 0.01 else ""}')

## Section 6: Approximate Centerpoint by Grid Search

To find the centerpoint, we'll search over a grid of candidate points and pick
the one with the highest Tukey depth. This is a brute-force approximation —
there are smarter algorithms, but this illustrates the idea.

In [ ]:
def approx_centerpoint(data, grid_n=50):
    """
    Approximate the centerpoint of a 2D dataset by exhaustive grid search.

    We evaluate Tukey depth at each point on a (grid_n x grid_n) grid
    spanning the bounding box of the data, then return the point of
    maximum depth.

    Parameters
    ----------
    data : array-like, shape (n, 2)
        The 2D dataset.
    grid_n : int
        Number of grid points along each axis (total: grid_n^2 evaluations).

    Returns
    -------
    best : ndarray, shape (2,)
        Approximate centerpoint (maximum Tukey depth).
    best_depth : float
        Tukey depth of the approximate centerpoint.
    """
    data = np.asarray(data, dtype=float)
    xmin, ymin = data.min(axis=0)
    xmax, ymax = data.max(axis=0)

    best = None
    best_depth = -1.0

    # Search over a regular grid spanning the bounding box
    for x in np.linspace(xmin, xmax, grid_n):
        for y in np.linspace(ymin, ymax, grid_n):
            d = tukey_depth(np.array([x, y]), data)
            if d > best_depth:
                best_depth = d
                best = np.array([x, y])

    return best, best_depth


# Apply to the 5-point example
cp5, depth5 = approx_centerpoint(points_5, grid_n=100)
print('Five-point example:')
print(f'  Approximate centerpoint: ({cp5[0]:.2f}, {cp5[1]:.2f})')
print(f'  Tukey depth at centerpoint: {depth5:.3f}')
print(f'  Required depth >= 1/3 = {1/3:.3f}: {depth5 >= 1/3 - 0.01}')
print()
print(f'  Coord. median depth: {depth_coord_median:.3f}')
print(f'  Centerpoint depth:   {depth5:.3f}')
print(f'  Improvement: {depth5 - depth_coord_median:.3f}')

## Section 7: Tukey Depth Heatmap

One of the most illuminating visualisations is a **depth heatmap**: we compute
the Tukey depth at every point on a grid and display it as a color image.

High-depth regions (red/warm) are the most central. The centerpoint sits at
the peak. The level sets of Tukey depth are convex regions — they shrink
inward toward the center like contour lines on a topographic map.

In [ ]:
def tukey_depth_heatmap(data, grid_n=25, title='Tukey Depth Heatmap', margin=0.5):
    """
    Compute and plot the Tukey depth heatmap for a 2D dataset.

    Parameters
    ----------
    data : ndarray, shape (n, 2)
    grid_n : int  — resolution of the depth grid
    title : str
    margin : float — fractional margin beyond the data range
    """
    data = np.asarray(data, dtype=float)
    xmin, ymin = data.min(axis=0)
    xmax, ymax = data.max(axis=0)

    # Add margin around the data range
    dx = (xmax - xmin) * margin + 0.5
    dy = (ymax - ymin) * margin + 0.5

    xs = np.linspace(xmin - dx, xmax + dx, grid_n)
    ys = np.linspace(ymin - dy, ymax + dy, grid_n)

    # Compute depth at every grid point
    depth_grid = np.zeros((grid_n, grid_n))
    for i, y in enumerate(ys):
        for j, x in enumerate(xs):
            depth_grid[i, j] = tukey_depth(np.array([x, y]), data)

    # Find the approximate centerpoint
    max_idx = np.unravel_index(np.argmax(depth_grid), depth_grid.shape)
    cp_x = xs[max_idx[1]]
    cp_y = ys[max_idx[0]]
    max_depth = depth_grid[max_idx]

    # Coordinatewise median
    coord_med = np.median(data, axis=0)

    # ---- Plot ----
    fig, ax = plt.subplots(figsize=(7, 6))

    # Display the depth as a heatmap
    extent = [xmin - dx, xmax + dx, ymin - dy, ymax + dy]
    im = ax.imshow(
        depth_grid,
        extent=extent,
        origin='lower',
        cmap='RdYlGn',    # Red (low depth) to green (high depth)
        aspect='auto',
        alpha=0.85
    )
    plt.colorbar(im, ax=ax, label='Tukey depth')

    # Overlay the data points
    ax.scatter(data[:, 0], data[:, 1], color='black', s=80, zorder=5, label='Data points')

    # Overlay coordinatewise median
    ax.scatter(*coord_med, color='blue', s=180, marker='D', zorder=6,
               label=f'Coord. median ({coord_med[0]:.1f}, {coord_med[1]:.1f})')

    # Overlay approximate centerpoint
    ax.scatter(cp_x, cp_y, color='white', edgecolors='black', s=220,
               marker='*', linewidths=1.5, zorder=7,
               label=f'Approx. centerpoint ({cp_x:.1f}, {cp_y:.1f})\ndepth = {max_depth:.3f}')

    # Add contour lines for specific depth levels
    ax.contour(xs, ys, depth_grid, levels=[1/3], colors='red',
               linestyles='--', linewidths=2)
    ax.plot([], [], color='red', linestyle='--', linewidth=2, label='depth = 1/3 (centerpoint threshold)')

    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=8, loc='upper right')

    plt.tight_layout()
    plt.show()

    return cp_x, cp_y, max_depth


# Apply to the 5-point example
print('Computing Tukey depth heatmap for the 5-point example...')
print('(This may take a few seconds — we are computing depth at 25×25 = 625 grid points)')
cp_x5, cp_y5, depth5 = tukey_depth_heatmap(
    points_5, grid_n=25,
    title='Tukey Depth Heatmap — Five-Point Example'
)

### Reading the Heatmap

- **Green regions** have high Tukey depth — they are "central" w.r.t. the data.
- **Red regions** have low Tukey depth — they are on the "edge" or outside.
- The **dashed red contour** marks depth $= 1/3$, which is the centerpoint threshold.
  The Centerpoint Theorem guarantees this region is non-empty!
- The **coordinatewise median** (blue diamond) may lie outside this region, while
  the **approximate centerpoint** (white star) should be inside or near it.

## Section 8: Application — Testing on the Five-Point Example and a Larger Dataset

Let's explicitly verify the Centerpoint Theorem's guarantee:
the approximate centerpoint should have depth $\geq 1/3$,
while the coordinatewise median may fail this.

In [ ]:
# ---- Verify on the 5-point example ----
coord_med5 = np.median(points_5, axis=0)
depth_cm5  = tukey_depth(coord_med5, points_5)
depth_cp5  = tukey_depth(np.array([cp_x5, cp_y5]), points_5)

print('Five-point example verification:')
print(f'  n = {len(points_5)},  threshold 1/3 = {1/3:.3f}')
print()
print(f'  Coordinatewise median = {coord_med5}')
print(f'  Tukey depth(coord. median) = {depth_cm5:.3f}   -> >= 1/3? {depth_cm5 >= 1/3 - 0.01}')
print()
print(f'  Approximate centerpoint = ({cp_x5:.2f}, {cp_y5:.2f})')
print(f'  Tukey depth(centerpoint) = {depth_cp5:.3f}   -> >= 1/3? {depth_cp5 >= 1/3 - 0.01}')

# ---- Now try a larger random dataset ----
np.random.seed(99)
big_data = np.random.randn(30, 2) * 10 + np.array([50, 50])

coord_med_big = np.median(big_data, axis=0)
depth_cm_big  = tukey_depth(coord_med_big, big_data)

cp_big, depth_cp_big = approx_centerpoint(big_data, grid_n=40)

print()
print('Random 30-point dataset:')
print(f'  Tukey depth(coord. median) = {depth_cm_big:.3f}  -> >= 1/3? {depth_cm_big >= 1/3 - 0.01}')
print(f'  Tukey depth(centerpoint)   = {depth_cp_big:.3f}  -> >= 1/3? {depth_cp_big >= 1/3 - 0.01}')

# Plot the larger dataset
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(big_data[:, 0], big_data[:, 1], color='steelblue', s=60, alpha=0.7, label='Data')
ax.scatter(*coord_med_big, color='darkorange', s=200, marker='D', zorder=5,
           label=f'Coord. median (depth={depth_cm_big:.3f})')
ax.scatter(*cp_big, color='red', s=200, marker='*', zorder=5,
           label=f'Approx. centerpoint (depth={depth_cp_big:.3f})')
ax.set_title('Coordinatewise Median vs Centerpoint — 30 Random Points', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 9: Helly's Theorem Demo

The Centerpoint Theorem is closely related to **Helly's Theorem**, one of the
most important results in combinatorial geometry.

> **Helly's Theorem (1923)**: Let $X_1, X_2, \ldots, X_m$ be a finite collection
> of convex sets in $\mathbb{R}^d$. If every $d+1$ of these sets have a common point,
> then all $m$ sets have a common point.

In 1D this says: if a collection of intervals on the number line has the property
that every **pair** of intervals overlaps, then they all share a common point.

Let's verify this for random intervals on $[0, 100]$!

In [ ]:
def helly_1d_demo(n_intervals=8, seed=7):
    """
    Demonstrate Helly's Theorem in 1D.

    Generates random intervals, checks if they are pairwise intersecting,
    and if so, finds their common intersection.

    In 1D: pairwise intersecting => all share a common point.
    The common intersection is [max of left endpoints, min of right endpoints].
    """
    np.random.seed(seed)

    # Generate random intervals [a, b] with a < b, all in [0, 100]
    lefts  = np.random.uniform(0, 60, n_intervals)
    rights = lefts + np.random.uniform(20, 60, n_intervals)  # Ensure b > a
    rights = np.minimum(rights, 100)  # Clip to [0, 100]
    intervals = list(zip(lefts, rights))

    print(f'Generated {n_intervals} random intervals:')
    for i, (l, r) in enumerate(intervals):
        print(f'  I_{i+1} = [{l:.1f}, {r:.1f}]')

    # Check pairwise intersections
    all_pairwise = True
    failing_pair = None
    for i in range(n_intervals):
        for j in range(i + 1, n_intervals):
            l_i, r_i = intervals[i]
            l_j, r_j = intervals[j]
            # Intervals [l_i, r_i] and [l_j, r_j] intersect iff l_j <= r_i and l_i <= r_j
            if not (l_j <= r_i and l_i <= r_j):
                all_pairwise = False
                failing_pair = (i, j)
                break

    print()
    print(f'Are all pairs of intervals intersecting? {all_pairwise}')

    if all_pairwise:
        # By Helly's theorem, the common intersection is non-empty
        common_left  = max(l for l, r in intervals)
        common_right = min(r for l, r in intervals)
        print(f'Helly guarantees a common point exists!')
        print(f'Common intersection: [{common_left:.1f}, {common_right:.1f}]')
        print(f'Common point (e.g. midpoint): {(common_left + common_right)/2:.1f}')
    else:
        i, j = failing_pair
        print(f'Intervals I_{i+1} and I_{j+1} do not intersect — Helly does not apply.')

    # ---- Plot ----
    fig, ax = plt.subplots(figsize=(10, 4))

    colors = plt.cm.tab10(np.linspace(0, 1, n_intervals))
    for i, ((l, r), color) in enumerate(zip(intervals, colors)):
        ax.barh(i, r - l, left=l, height=0.5, color=color, alpha=0.7,
                label=f'$I_{{{i+1}}}$ = [{l:.1f}, {r:.1f}]')
        ax.text(l + (r - l)/2, i, f'$I_{{{i+1}}}$', ha='center', va='center',
                fontsize=9, fontweight='bold', color='black')

    if all_pairwise:
        ax.axvspan(common_left, common_right, alpha=0.25, color='red',
                   label=f'Common intersection [{common_left:.1f}, {common_right:.1f}]')
        ax.axvline(x=(common_left + common_right)/2, color='red',
                   linestyle='--', linewidth=2, label='Common point')

    ax.set_xlabel('Position on number line', fontsize=12)
    ax.set_ylabel('Interval index', fontsize=12)
    ax.set_yticks(range(n_intervals))
    ax.set_yticklabels([f'$I_{{{i+1}}}$' for i in range(n_intervals)])
    ax.set_title("Helly's Theorem in 1D: pairwise-intersecting intervals → common point",
                 fontsize=12)
    ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()


helly_1d_demo(n_intervals=8, seed=42)

### Why Does Helly's Theorem Relate to Centerpoints?

The proof of the Centerpoint Theorem uses Helly's Theorem as a key ingredient!
Here's the idea:

For each point $p_i$ in your dataset, define a **convex set** $C_i$ as the
region that is "covered" by at least $2n/3$ data points on each side.

Helly's Theorem guarantees that if every 3 of these sets share a common point,
then ALL of them do — and that common point is the centerpoint!

This is a beautiful example of how different theorems in geometry connect
together into a coherent theory.

## Summary and Key Takeaways

In this notebook we explored **Tukey depth** and the **Centerpoint Theorem**.

### What We Learned

1. **The mean fails under outliers**: one extreme value can drag the mean far
   from the bulk of the data.

2. **The coordinatewise median** is more robust but is NOT guaranteed to have
   a large fraction of points on each side in 2D.

3. **Tukey halfspace depth**: $\text{depth}(p, X) = $ minimum fraction of points
   in any halfspace containing $p$. Measures how "central" a point is.

4. **The Centerpoint Theorem**: for $n$ points in $\mathbb{R}^2$, there always
   exists a point $c$ with Tukey depth $\geq 1/3$.

5. **Helly's Theorem** (1D demonstration): pairwise intersecting intervals
   always share a common point. Used in the proof of the Centerpoint Theorem.

### Further Exploration

- For $d=3$ (3D points), the Centerpoint Theorem guarantees depth $\geq n/4$. Why $1/(d+1)$?
- The **Tukey median** (deepest point) can be thought of as the true "centre" of a
  multivariate dataset. It is used in robust statistics.
- Research the **ham sandwich theorem**: a single hyperplane can simultaneously
  bisect any $d$ datasets in $\mathbb{R}^d$!